<a href="https://colab.research.google.com/github/suyogy1-bot/Numpy_Pandas/blob/main/Class_notes/netflix_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Netflix Movies Dataset — Data Cleaning Project
## Complete Solution Notebook

---

**Topic:** Real-World Data Cleaning with NumPy and Pandas  
**Dataset:** `netflix_movies_dirty (1).csv`

---

### What you will learn in this notebook

By the end of this project you will be able to:

1. Load a messy, real-world dataset and understand its shape and structure
2. Identify every type of data quality problem a dataset can have
3. Write Python functions that clean data step by step
4. Apply those functions to a full DataFrame using `.apply()`
5. Detect and handle missing values, duplicates, wrong data types, and outliers
6. Produce a clean, analysis-ready dataset

---

> **How to read this notebook:** Every section follows this pattern:
> - **Problem Found** — what we discovered
> - **Why is this a problem?** — the real-world impact
> - **How can we solve it?** — the plan before we code
> - Code — beginner-friendly solution
> - Explanation — what just happened

Let's begin.

---
# Step 1 — Import Libraries and Load the Dataset

Before we do anything, we import the libraries we will use throughout the project.


In [161]:
import pandas as pd
import numpy as np

load_df = pd.read_csv("netflix_movies_dirty.csv")
df = load_df.copy()

In [162]:
df

,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House
0,NF2602,Little Women!!!,animation,1996,3h 13m,4.7,9.5,317073,Patty Jenkins,Japan,Hin,-500000,1505957457,2015-10-14,R,"Leonardo DiCaprio, Jodie Foster",Pixar Animation Studios
1,NF3430,The White Tiger,"Animation, Drama",2007,1h 27m,8.4,4.0,123876,Jane Campion,U.S.,Kor,210868644,"687,371,362",20220620,R,"Sandra Bullock, Robert De Niro, Ryan Reynolds,...",Netflix Originals
2,NF5757,His House (Extended Collector's Edition with B...,ROMANCE,1993,1h 57m,7.4,3.4,"1,558,685",Spike Lee,India,korean,100147805,478302873,20180120,G,"Robert Downey Jr., Meryl Streep, Michael B. Jo...",Paramount Pictures
3,NF8495,Roma,"Thriller, Drama",2004,160mins,7.6,7.0,"46,529",NaN,France,KOREAN,95587293,1295144836,20210103,G,NaN,New Line Cinema
4,NF7790,Fear Street Part Three 1666,Romance,2016,113 min,5.0,5.7,682718,NaN,South Korea,French,8921122,687779882,2021-12-08,PG-13,"Timothée Chalamet, Oscar Isaac, Cate Blanchett...",Lionsgate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1555,NF4901,The Hand of God,Drama,2017,135mins,8.3,5.0,1525690,James Gunn,France,KOREAN,NaN,"730,299,883",28/10/2015,NR,"Lupita Nyong'o, Margot Robbie, Pedro Pascal, L...",DC Films
1556,NF4202,Mass,Thriller,1996,121 min,6.4,5.8,1779958,Spike Lee,U.K.,ENGLISH,$61305141,42703073,"November 26, 2022",R,"Pedro Pascal, Brad Pitt, Al Pacino",Apple TV+
1557,NF9751,NaN,THRILLER,2016,1h 43m,6.8,7.1,768218,Francis Ford Coppola,Italy,ENGLISH,"$80,210,058","1,724,230,587","May 19, 2018",NC-17,"Al Pacino, Angelina Jolie",Marvel Studios
1558,NF4195,Demon Slayer Mugen Train,ACTION,2017,NaN,4.5,-1.0,141444,Ryan Coogler,South Korea,FRENCH,202601908,NaN,12/11/2021,PG-13,"Scarlett Johansson, Lupita Nyong'o",Columbia Pictures


---
# Step 2 — First Look at the Dataset

A real data analyst never jumps straight into cleaning. First, we look at what we have.


In [163]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1560 non-null   object 
 1   Title             1483 non-null   object 
 2   Genre             1560 non-null   object 
 3   Release_Year      1560 non-null   int64  
 4   Duration          1383 non-null   object 
 5   Rating            1434 non-null   float64
 6   IMDb_Rating       1394 non-null   float64
 7   Votes             1480 non-null   object 
 8   Director          1488 non-null   object 
 9   Country           1560 non-null   object 
 10  Language          1560 non-null   object 
 11  Budget            1438 non-null   object 
 12  Revenue           1405 non-null   object 
 13  Date_Added        1560 non-null   object 
 14  Age_Rating        1560 non-null   object 
 15  Cast              1458 non-null   object 
 16  Production_House  1486 non-null   object 


In [164]:
df.shape

(1560, 17)

In [165]:
for i,k in enumerate(df.columns):
  print(i, k)

0 Movie_ID
1 Title
2 Genre
3 Release_Year
4 Duration
5 Rating
6 IMDb_Rating
7 Votes
8 Director
9 Country
10 Language
11 Budget
12 Revenue
13 Date_Added
14 Age_Rating
15 Cast
16 Production_House


In [166]:
df.describe()

,Release_Year,Rating,IMDb_Rating
count,1560.000000,1434.000000,1394.000000
mean,2006.794872,6.766109,6.395481
std,27.353181,1.614089,2.889410
min,1890.000000,4.000000,-1.000000
25%,1998.000000,5.300000,4.525000
50%,2007.000000,6.800000,6.300000
75%,2017.000000,8.200000,8.200000
max,2099.000000,9.500000,15.000000


Notice that **every column shows `object` (string) as its dtype** — that is expected, because we loaded everything as text. Our job is to fix the types after we clean the values.

Now let's look for problems systematically.


---
# Step 3 — Finding Missing Values

## **Problem Found**

Some cells in the dataset appear empty. We need to find exactly how many are missing and in which columns.

## **Why is this a problem?**

Missing values cause errors when you try to do calculations or analysis. For example, you cannot compute the average IMDb rating if some values are empty.

## **How can we solve it?**

We use `.isnull().sum()` to count missing values column by column. We also check for **empty strings**, because sometimes a cell looks empty but actually contains `""` — which pandas does not consider as `NaN`.

In [167]:
df.isnull().sum()

,0
Movie_ID,0
Title,77
Genre,0
Release_Year,0
Duration,177
Rating,126
IMDb_Rating,166
Votes,80
Director,72
Country,0


We can see that many columns have missing or empty values. We will deal with each one when we reach that column.

For now, let's continue discovering other problems first.


---
# Step 4 — Checking for Duplicate Movie IDs

## **Problem Found**

Each movie should have a **unique ID**. Let's check if any Movie_IDs appear more than once.

## **Why is this a problem?**

If two different rows share the same ID, we cannot tell them apart. Any analysis that relies on Movie_ID (like joining tables) will produce wrong results.

## **How can we solve it?**

We count how many unique IDs we have and compare it to the total number of rows. We also find which IDs appear more than once.

In [168]:
# Total rows vs unique Movie_IDs

duplicate_ids = df['Movie_ID'].nunique()
duplicate_ids

1316

In [169]:
# Find which Movie_ID appears more than once.

movie_id_counts = df['Movie_ID'].value_counts()

In [170]:
movie_id_counts[movie_id_counts > 1]

,count
Movie_ID,
NF2666,4
NF5819,4
NF6886,4
NF9963,3
NF7033,3
...,...
NF8903,2
NF9140,2
NF9252,2


## 💻 Solution — Remove Rows with Duplicate Movie IDs

We will keep the **first** occurrence of each Movie_ID and remove the rest.

> **Note:** In a real job, you might investigate each duplicate carefully. Here, we take the safe approach of keeping the first record.


In [171]:
df.drop_duplicates(subset="Movie_ID", keep="first", inplace=True)

In [172]:
df.shape

(1316, 17)

✅ **Explanation:** `drop_duplicates(subset="Movie_ID", keep="first")` looks at the Movie_ID column only and removes every row that has the same ID as a row that appeared earlier. The first occurrence is kept.


Now let's write a function to clean each title:


## Handling Rows with Missing Titles

## **Why is this a problem?**

A movie with no title is useless in the dataset — we cannot identify it. We should remove those rows.


In [173]:
# find Title values which are null

df[df['Title'].isna()]

,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House
34,NF5158,NaN,Drama,2024,167mins,4.7,6.8,942833,Alfonso Cuaron,Germany,french,173928457,1073775632,2015-07-27,R,"Denzel Washington, Will Smith",Walt Disney Pictures
37,NF6194,NaN,Animation,1994,161mins,6.8,3.5,477708,Ridley Scott,South Korea,english,$181722742,650176760,"April 23, 2019",PG-13,"Robert Downey Jr., Leonardo DiCaprio, Dwayne J...",Marvel Studios
70,NF9575,NaN,Comedy | Thriller,2009,128,6.0,8.4,NaN,Tim Burton,Japan,japanese,72372322,1882372841,20201201,PG,"Tom Hanks, Dwayne Johnson, Ana de Armas",Blumhouse Productions
105,NF9074,NaN,Sci-Fi,1998,192,7.7,3.8,NaN,Bong Joon-ho,U.S.A,JAPANESE,"$198,869,279",1976770552,2023-11-20,TV-PG,"Scarlett Johansson, Julia Roberts",Columbia Pictures
108,NF5670,NaN,Thriller/Comedy,1995,126 min,7.0,4.1,196490,James Cameron,Germany,Japanese,113208537,1135853858,"August 16, 2017",TV-MA,"Margot Robbie, Timothée Chalamet, Robert De Niro",DC Films
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,NF9667,NaN,Fantasy,2019,169mins,5.5,6.1,1876871,Francis Ford Coppola,South Korea,GERMAN,117695199,1014116288,20190713,NR,"Cate Blanchett, Florence Pugh, Will Smith",Columbia Pictures
1537,NF6245,NaN,"Adventure,",2018,NaN,6.1,6.7,1610699,Bong Joon-ho,France,Hin,160083811,1688237966,2021-08-21,TV-14,"Michael B. Jordan, Jodie Foster, Oscar Isaac",HBO Films
1550,NF3575,NaN,Comedy,1990,88,5.7,-1.0,599610,Ridley Scott,India,Spanish,178067761,1508613733,2017-04-26,TV-MA,"Cate Blanchett, Leonardo DiCaprio",Universal Pictures
1557,NF9751,NaN,THRILLER,2016,1h 43m,6.8,7.1,768218,Francis Ford Coppola,Italy,ENGLISH,"$80,210,058","1,724,230,587","May 19, 2018",NC-17,"Al Pacino, Angelina Jolie",Marvel Studios


In [174]:
# df = df.dropna(subset=['Title'])

In [175]:
# df[df['Title'].isna()]

✅ **Explanation:** We first clean every title (removing whitespace, newlines, etc.). Any title that becomes empty after cleaning is set to `NaN`. Then we drop those rows entirely because a movie without a name cannot be analysed.


---
# Step 5 — Cleaning the Title Column

## **Problem Found**

The `Title` column has many problems:

1. **Empty titles** — some rows have no title at all
2. **Extra whitespace** — leading spaces, trailing spaces, double spaces
3. **Newline characters** (`\n`) — titles end with invisible line breaks
4. **ALL CAPS titles** — some titles are fully uppercased
5. **Exclamation marks** — titles like `"Inception!!!"`
6. **Excessively long titles** — titles like `"His House (Extended Collector's Edition with Bonus Features and Director's Commentary)"`

## **Why is this a problem?**

If two rows have the title `"Inception"` and `"  inception\n"`, they look like different movies to a computer even though they are the same.

## **How can we solve it?**

We will write a function that handles each problem one by one. Then we apply that function to every title in the dataset.

In [176]:
# convert to string

# df['Title'] = df['Title'].astype(str)

In [177]:
# # Remove newlines and tabs from every Title

# df['Title'] = df['Title'].str.replace('\r', ' ', regex = False)
# df['Title'] = df['Title'].str.replace('\n', ' ', regex = False)
# df['Title'] = df['Title'].str.replace('\t', ' ', regex = False)


In [178]:
# Strip leading/trailing spaces

# df['Title'] = df['Title'].str.strip()

In [179]:
# Collapse multiple spaces into a single space

# df['Title'] = df['Title'].str.split().str.join(' ')

In [180]:
# Turn empty strings into NaN

# df['Title'] = df['Title'].replace('', pd.NA)

In [181]:
# Drop rows where Title is now NaN

# df = df.dropna(subset=['Title'])

In [182]:
df['Title']

,Title
0,Little Women!!!
1,The White Tiger
2,His House (Extended Collector's Edition with B...
3,Roma
4,Fear Street Part Three 1666
...,...
1555,The Hand of God
1556,Mass
1557,NaN
1558,Demon Slayer Mugen Train


In [183]:
def clean_title(title):


  # Task 1: Handle missing/NaN

  if pd.isna(title):
    return np.nan

  # Task 2: Remove /n or /t

  title = title.replace('\r', ' ')
  title = title.replace('\n', ' ')
  title = title.replace('\t', ' ')

  # Task 3: Remove extra whitespace

  title = title.strip()
  title = ' '.join(title.split())

  # Task 4: Remove punctuation like !!!

  title = title.rstrip('!')

  # Task 5: Truncate long titles
  if '(' in title:
    title = title[:title.index('(')].strip()

  # Task 6: Convert to title case

  title = title.title()

  return title

In [184]:
df['Title'].unique()

array(['Little Women!!!', '  The White Tiger  ',
       "His House (Extended Collector's Edition with Bonus Features and Director's Commentary)",
       'Roma', 'Fear Street Part Three 1666', 'Another Round',
       'OUTER BANKS', 'Space Jam A New Legacy', 'Candyman',
       'The Lost Daughter', 'Hubie Halloween', 'Greyhound', 'Flee ',
       'THE ONE AND ONLY IVAN', 'The Old Guard', 'His House',
       'Hell or High Water', 'Extraction', 'Army of the Dead',
       'Bad Boys for Life', 'The Little Things', 'The Addams Family 2',
       'Black Widow', 'Joker ', 'Sound of Metal', 'Home Sweet Home Alone',
       'Get Out', 'Malignant', 'The Humans', 'Bridgerton',
       'Project Power', 'The Invisible Man', 'Phantom Thread', 'SOUL',
       nan, 'Whiplash', 'Time', 'Bird Box', 'No Country for Old Men',
       'Summer of Soul', 'Fight Club', 'Halloween Kills',
       'A Boy Called Christmas', 'Judas and the Black Messiah', 'Titane',
       'A Castle for Christmas', 'Coming 2 America',
     

In [185]:
df['Title'] = df['Title'].apply(clean_title)

In [186]:
df['Title']

,Title
0,Little Women
1,The White Tiger
2,His House
3,Roma
4,Fear Street Part Three 1666
...,...
1555,The Hand Of God
1556,Mass
1557,NaN
1558,Demon Slayer Mugen Train


✅ **Explanation:** The key idea is to pick **one separator** and split on it. We convert all separators (`,`, `|`, `/`) into `|`, then take only the part before the first `|`. This gives us the primary genre. Title Case makes every genre consistent.


---
# Step 6 — Standardising the Genre Column

## **Problem Found**

The `Genre` column has many inconsistencies:

- `"action"`, `"ACTION"`, `"Action"` — same genre, different casing
- `"Action "`, `" Fantasy"`, `"\tDrama"` — leading/trailing spaces and tab characters
- `"Action, Drama"`, `"Action | Drama"`, `"Drama/Comedy"` — multiple genres joined with different separators

## **Why is this a problem?**

If you try to count how many action movies there are, `"action"`, `"ACTION"`, and `"Action "` will all be counted separately. You will get the wrong answer.

## **How can we solve it?**

We will extract only the **first genre** from each row (the primary genre), then standardise its casing and remove extra whitespace. This gives us one clean category per movie.

In [187]:
df['Genre']

,Genre
0,animation
1,"Animation, Drama"
2,ROMANCE
3,"Thriller, Drama"
4,Romance
...,...
1555,Drama
1556,Thriller
1557,THRILLER
1558,ACTION


In [188]:
df['Genre'].unique()

array(['animation', 'Animation, Drama', 'ROMANCE', 'Thriller, Drama',
       'Romance', 'Drama | Thriller', 'Romance ', 'COMEDY', 'Comedy,',
       'Horror  ', 'ACTION', 'Adventure,', 'Crime', 'Drama, Drama',
       'Animation | Thriller', 'Fantasy ', 'adventure', ' Fantasy',
       'Fantasy', 'comedy', 'Crime | Thriller', 'crime', 'Romance, Drama',
       ' Adventure', 'Sci-Fi, Drama', 'Documentary', 'Comedy | Thriller',
       'Animation  ', 'Comedy', ' Drama', 'Fantasy  ', 'ANIMATION',
       'Animation', 'Sci-Fi/Comedy', 'Documentary, Drama', 'Romance,',
       'HORROR', 'Action,', ' Romance', ' Horror', 'Documentary  ',
       'Animation ', 'Thriller', 'Action/Comedy', 'Sci-Fi',
       'Thriller | Thriller', 'action', 'Comedy, Drama', 'Documentary ',
       'ADVENTURE', 'Crime  ', 'documentary', 'Comedy ', 'Sci-Fi,',
       ' Thriller', 'fantasy', ' Comedy', 'Crime,', 'Drama ',
       'Animation,', 'Fantasy, Drama', ' Sci-Fi', 'Action ', 'Drama  ',
       'Sci-Fi ', 'Adventure', '

In [189]:
def clean_genre(genre):

  # Task 1: Handle missing/NaN

  if pd.isna(genre):
    return np.nan

  # Task 2: Remove \t or \n

  genre = genre.replace('\t', '').replace('\n', '')

  # Task 3: Extract only first genre if there are 2 present

  for sep in [',', '|', '/']:
        if sep in genre:
            genre = genre.split(sep, 1)[0]
            break

  # Task 4: Removing extra whitespaces

  genre = genre.strip()
  genre = ' '.join(genre.split())

  # Task 5: Convert to title case

  genre = genre.title()

  return genre

In [190]:
df['Genre'] = df['Genre'].apply(clean_genre)

In [191]:
df['Genre'].unique()

array(['Animation', 'Romance', 'Thriller', 'Drama', 'Comedy', 'Horror',
       'Action', 'Adventure', 'Crime', 'Fantasy', 'Sci-Fi', 'Documentary'],
      dtype=object)

✅ **Explanation:** We set a sensible boundary: 1900 to 2025. Any value outside that range is treated as an error and replaced with `NaN`. We do not guess the correct year because we have no way of knowing what it should be.


---
# Step 7 — Fixing the Release Year Column

## **Problem Found**

The `Release_Year` column contains **outlier values** that are impossible or unrealistic:

- `1890` — movies did not exist in 1890
- `2099` — that is in the future

## **Why is this a problem?**

If we calculate the average release year, impossible values will pull the result in the wrong direction. We also cannot correctly sort movies by year.

## **How can we solve it?**

We convert the column to numbers. Then we set any year outside a realistic range (1900–2025) to `NaN` (missing), so they are excluded from analysis.

In [192]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1316 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1316 non-null   object 
 1   Title             1251 non-null   object 
 2   Genre             1316 non-null   object 
 3   Release_Year      1316 non-null   int64  
 4   Duration          1164 non-null   object 
 5   Rating            1212 non-null   float64
 6   IMDb_Rating       1180 non-null   float64
 7   Votes             1247 non-null   object 
 8   Director          1255 non-null   object 
 9   Country           1316 non-null   object 
 10  Language          1316 non-null   object 
 11  Budget            1213 non-null   object 
 12  Revenue           1182 non-null   object 
 13  Date_Added        1316 non-null   object 
 14  Age_Rating        1316 non-null   object 
 15  Cast              1235 non-null   object 
 16  Production_House  1252 non-null   object 
dtype

In [193]:
df['Release_Year'].min()

1890

In [194]:
df['Release_Year'].max()

2099

In [195]:
lower = (df['Release_Year'] < 1900).sum()
lower

np.int64(38)

In [196]:
upper = (df['Release_Year'] > 2026).sum()
upper

np.int64(45)

In [197]:
# Step 7 — Fixing the Release Year Column

# 1) Ensure the column is numeric
df['Release_Year'] = pd.to_numeric(df['Release_Year'], errors='coerce')


In [198]:
# Find why we are converting int into to_numeric float value?

In [199]:
def clean_year(year):

  if year < 1900 or year > 2025:
    return np.nan
  else:
    return year

In [200]:
# What if we didn't user else statement, then why the code give NaN values in
# every field?

In [201]:
df['Release_Year'] = df['Release_Year'].apply(clean_year)
df['Release_Year'] = df['Release_Year'].astype('Int64')
df

,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House
0,NF2602,Little Women,Animation,1996,3h 13m,4.7,9.5,317073,Patty Jenkins,Japan,Hin,-500000,1505957457,2015-10-14,R,"Leonardo DiCaprio, Jodie Foster",Pixar Animation Studios
1,NF3430,The White Tiger,Animation,2007,1h 27m,8.4,4.0,123876,Jane Campion,U.S.,Kor,210868644,"687,371,362",20220620,R,"Sandra Bullock, Robert De Niro, Ryan Reynolds,...",Netflix Originals
2,NF5757,His House,Romance,1993,1h 57m,7.4,3.4,"1,558,685",Spike Lee,India,korean,100147805,478302873,20180120,G,"Robert Downey Jr., Meryl Streep, Michael B. Jo...",Paramount Pictures
3,NF8495,Roma,Thriller,2004,160mins,7.6,7.0,"46,529",NaN,France,KOREAN,95587293,1295144836,20210103,G,NaN,New Line Cinema
4,NF7790,Fear Street Part Three 1666,Romance,2016,113 min,5.0,5.7,682718,NaN,South Korea,French,8921122,687779882,2021-12-08,PG-13,"Timothée Chalamet, Oscar Isaac, Cate Blanchett...",Lionsgate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1555,NF4901,The Hand Of God,Drama,2017,135mins,8.3,5.0,1525690,James Gunn,France,KOREAN,NaN,"730,299,883",28/10/2015,NR,"Lupita Nyong'o, Margot Robbie, Pedro Pascal, L...",DC Films
1556,NF4202,Mass,Thriller,1996,121 min,6.4,5.8,1779958,Spike Lee,U.K.,ENGLISH,$61305141,42703073,"November 26, 2022",R,"Pedro Pascal, Brad Pitt, Al Pacino",Apple TV+
1557,NF9751,NaN,Thriller,2016,1h 43m,6.8,7.1,768218,Francis Ford Coppola,Italy,ENGLISH,"$80,210,058","1,724,230,587","May 19, 2018",NC-17,"Al Pacino, Angelina Jolie",Marvel Studios
1558,NF4195,Demon Slayer Mugen Train,Action,2017,NaN,4.5,-1.0,141444,Ryan Coogler,South Korea,FRENCH,202601908,NaN,12/11/2021,PG-13,"Scarlett Johansson, Lupita Nyong'o",Columbia Pictures


In [202]:
#  2) Set unrealistic years to NaN (outside 1900–2025)

# df.loc[(df['Release_Year'] < 1900) | (df['Release_Year'] > 2025), 'Release_Year'] = np.nan

In [203]:
df['Release_Year'].unique()

<IntegerArray>
[1996, 2007, 1993, 2004, 2016, 2021, 2012, 2022, 2024, 2023, 1994, 2014, 2017,
 2006, 2001, 1999, 2002, 2000, 1995, <NA>, 2003, 1990, 2010, 2009, 1998, 2013,
 2020, 2019, 1991, 1997, 2015, 1992, 2011, 2005, 2008, 2018]
Length: 36, dtype: Int64

✅ **Explanation:** The key insight is that we handle each format with a separate `if-elif` block. The `"h"` character is our detector for the hours-and-minutes format. For everything else, we extract only the digits. The outlier filter (30–600 minutes) removes impossible values.


---
# Step 9 — Converting the Rating Column

## **Problem Found**

The `Rating` column contains numbers but they are **stored as text (strings)** because we loaded the file with `dtype=str`. Some values are also missing.

## **Why is this a problem?**

You cannot do arithmetic on text. `"8.5" + "7.0"` is `"8.57.0"` in Python, not `15.5`.

## **How can we solve it?**

We use `pd.to_numeric()` to convert the column to floating-point numbers. The `errors="coerce"` argument automatically turns any non-numeric value into `NaN`.

In [204]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1316 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1316 non-null   object 
 1   Title             1251 non-null   object 
 2   Genre             1316 non-null   object 
 3   Release_Year      1233 non-null   Int64  
 4   Duration          1164 non-null   object 
 5   Rating            1212 non-null   float64
 6   IMDb_Rating       1180 non-null   float64
 7   Votes             1247 non-null   object 
 8   Director          1255 non-null   object 
 9   Country           1316 non-null   object 
 10  Language          1316 non-null   object 
 11  Budget            1213 non-null   object 
 12  Revenue           1182 non-null   object 
 13  Date_Added        1316 non-null   object 
 14  Age_Rating        1316 non-null   object 
 15  Cast              1235 non-null   object 
 16  Production_House  1252 non-null   object 
dtype

In [205]:
df['Rating'].isna().sum()

np.int64(104)

✅ **Explanation:** `pd.to_numeric(errors="coerce")` is the cleanest way to convert a text column to numbers when you know some values might not be valid numbers. Any value that cannot become a number silently becomes `NaN`.


---
# Step 10 — Cleaning the IMDb_Rating Column

## **Problem Found**

The IMDb rating scale runs from **0 to 10**. Our dataset contains values outside this range:

- Values **greater than 10** (e.g. `15.0`) — impossible on IMDb
- Values **below 0** (e.g. `-1.0`) — also impossible

## **Why is this a problem?**

These outliers will corrupt any analysis involving IMDb ratings. The maximum, average, and sorting will all be wrong.

## **How can we solve it?**

1. Convert the column to floats
2. Replace any value outside `[0, 10]` with `NaN`

In [206]:
df['IMDb_Rating'].min()

-1.0

In [207]:
df['IMDb_Rating'].max()

15.0

In [208]:
def clean_IMDb_Rating(rating):

  if pd.isna(rating):
    return np.nan

  if rating < 0 or rating > 10:
    return np.nan
  else:
    return rating

In [209]:
df['IMDb_Rating'] = df['IMDb_Rating'].apply(clean_IMDb_Rating)
df['IMDb_Rating']

,IMDb_Rating
0,9.5
1,4.0
2,3.4
3,7.0
4,5.7
...,...
1555,5.0
1556,5.8
1557,7.1
1558,NaN


✅ **Explanation:** The IMDb rating range (0–10) is a known domain rule. Any value outside that range is a data entry error, so we set it to `NaN`. This is called **domain-rule validation**.


#Formatting Duration column: 1h 5m -> 65 mins

In [210]:
df

,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House
0,NF2602,Little Women,Animation,1996,3h 13m,4.7,9.5,317073,Patty Jenkins,Japan,Hin,-500000,1505957457,2015-10-14,R,"Leonardo DiCaprio, Jodie Foster",Pixar Animation Studios
1,NF3430,The White Tiger,Animation,2007,1h 27m,8.4,4.0,123876,Jane Campion,U.S.,Kor,210868644,"687,371,362",20220620,R,"Sandra Bullock, Robert De Niro, Ryan Reynolds,...",Netflix Originals
2,NF5757,His House,Romance,1993,1h 57m,7.4,3.4,"1,558,685",Spike Lee,India,korean,100147805,478302873,20180120,G,"Robert Downey Jr., Meryl Streep, Michael B. Jo...",Paramount Pictures
3,NF8495,Roma,Thriller,2004,160mins,7.6,7.0,"46,529",NaN,France,KOREAN,95587293,1295144836,20210103,G,NaN,New Line Cinema
4,NF7790,Fear Street Part Three 1666,Romance,2016,113 min,5.0,5.7,682718,NaN,South Korea,French,8921122,687779882,2021-12-08,PG-13,"Timothée Chalamet, Oscar Isaac, Cate Blanchett...",Lionsgate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1555,NF4901,The Hand Of God,Drama,2017,135mins,8.3,5.0,1525690,James Gunn,France,KOREAN,NaN,"730,299,883",28/10/2015,NR,"Lupita Nyong'o, Margot Robbie, Pedro Pascal, L...",DC Films
1556,NF4202,Mass,Thriller,1996,121 min,6.4,5.8,1779958,Spike Lee,U.K.,ENGLISH,$61305141,42703073,"November 26, 2022",R,"Pedro Pascal, Brad Pitt, Al Pacino",Apple TV+
1557,NF9751,NaN,Thriller,2016,1h 43m,6.8,7.1,768218,Francis Ford Coppola,Italy,ENGLISH,"$80,210,058","1,724,230,587","May 19, 2018",NC-17,"Al Pacino, Angelina Jolie",Marvel Studios
1558,NF4195,Demon Slayer Mugen Train,Action,2017,NaN,4.5,NaN,141444,Ryan Coogler,South Korea,FRENCH,202601908,NaN,12/11/2021,PG-13,"Scarlett Johansson, Lupita Nyong'o",Columbia Pictures


In [211]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1316 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1316 non-null   object 
 1   Title             1251 non-null   object 
 2   Genre             1316 non-null   object 
 3   Release_Year      1233 non-null   Int64  
 4   Duration          1164 non-null   object 
 5   Rating            1212 non-null   float64
 6   IMDb_Rating       1092 non-null   float64
 7   Votes             1247 non-null   object 
 8   Director          1255 non-null   object 
 9   Country           1316 non-null   object 
 10  Language          1316 non-null   object 
 11  Budget            1213 non-null   object 
 12  Revenue           1182 non-null   object 
 13  Date_Added        1316 non-null   object 
 14  Age_Rating        1316 non-null   object 
 15  Cast              1235 non-null   object 
 16  Production_House  1252 non-null   object 
dtype

In [212]:
def clean_duration(value):
    if pd.isna(value):
        return np.nan

    # 1. Normalise text
    text = str(value).lower().strip()


    # 2. Just digits → minutes
    if text.isdigit():
        minutes = int(text)

    # 3. Has hours, like "1h 57m" or "1h"
    elif "h" in text:
        parts = text.split("h")
        h_part = parts[0].strip()
        m_part = parts[1].strip() if len(parts) > 1 else ""

        hours = int(h_part)

        m_part = (
            m_part.replace("mins", "")
                 .replace("min", "")
                 .replace("m", "")
                 .strip()
        )

        if m_part == "":
            minutes = hours * 60
        else:
            minutes = hours * 60 + int(m_part)

    # 4. Otherwise treat as minutes with suffix, like "160mins", "45m"
    else:
        m_part = (
            text.replace("mins", "")
                .replace("min", "")
                .replace("m", "")
                .strip()
        )
        if m_part.isdigit():
            minutes = int(m_part)
        else:
            return np.nan

    # Optional sanity filter
    if minutes < 30 or minutes > 600:
        return np.nan

    return minutes

df["Duration"] = df["Duration"].apply(clean_duration).astype("Int64")


In [213]:
df['Duration']

,Duration
0,193
1,87
2,117
3,160
4,113
...,...
1555,135
1556,121
1557,103
1558,<NA>


In [214]:
df['Duration'].unique()

<IntegerArray>
[193,  87, 117, 160, 113, 171, 115, 151,  78, 122,
 ...
  79, 153, 164,  83, 152, 118, 199, 166, 157, 140]
Length: 132, dtype: Int64

---
# Step 11 — Cleaning the Votes Column

## **Problem Found**

The `Votes` column contains numbers, but some are formatted with **commas** (e.g. `"1,558,685"`). Pandas treats these as text, not numbers.

## **Why is this a problem?**

`"1,558,685"` is a string. You cannot sort or sum strings that look like numbers.

## **How can we solve it?**

We write a function that removes the commas and converts the value to an integer.

In [215]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1316 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1316 non-null   object 
 1   Title             1251 non-null   object 
 2   Genre             1316 non-null   object 
 3   Release_Year      1233 non-null   Int64  
 4   Duration          1112 non-null   Int64  
 5   Rating            1212 non-null   float64
 6   IMDb_Rating       1092 non-null   float64
 7   Votes             1247 non-null   object 
 8   Director          1255 non-null   object 
 9   Country           1316 non-null   object 
 10  Language          1316 non-null   object 
 11  Budget            1213 non-null   object 
 12  Revenue           1182 non-null   object 
 13  Date_Added        1316 non-null   object 
 14  Age_Rating        1316 non-null   object 
 15  Cast              1235 non-null   object 
 16  Production_House  1252 non-null   object 
dtype

In [216]:
df['Votes']

,Votes
0,317073
1,123876
2,"1,558,685"
3,"46,529"
4,682718
...,...
1555,1525690
1556,1779958
1557,768218
1558,141444


In [217]:
a = "1,558,685"
b = a.split(',')

In [218]:
c = "".join(b)
c

'1558685'

In [219]:
d = int(c)
d
type(d)

int

In [220]:
def clean_votes(votes):
  if pd.isnull(votes):
    return np.nan

  votes = "".join(votes.split(','))
  return int(votes)


df['Votes'] = df['Votes'].apply(clean_votes)

In [221]:
df['Votes']

,Votes
0,317073.0
1,123876.0
2,1558685.0
3,46529.0
4,682718.0
...,...
1555,1525690.0
1556,1779958.0
1557,768218.0
1558,141444.0


✅ **Explanation:** The `.replace(",", "")` removes the comma separators. Then `.isdigit()` confirms the remaining characters are all digits before we convert to `int`.


In [222]:
# Another way of solving this problem:


def clean_votes(v):
    # missing values
    if pd.isna(v):
        return np.nan

    # make sure it's a string
    text = str(v).strip()

    # remove comma separators
    text = text.replace(",", "")

    # if after removing commas it's all digits → safe to convert
    if text.isdigit():
        return int(text)
    else:
        return np.nan

In [223]:
a = '157987234'
a.isdigit() # This does not work on int values

True

---
# Step 12 — Cleaning the Budget Column

## **Problem Found**

The `Budget` column has two problems:

1. **Dollar signs and commas** — e.g. `"$176,890,118"` is text, not a number
2. **Negative values** — e.g. `"-500000"` — a budget cannot be negative

## **Why is this a problem?**

We cannot do any financial analysis (like comparing budget to revenue) until budget is a clean positive number.

## **How can we solve it?**

Remove the `$` and `,`, convert to a number, then set negative values to `NaN`.

In [224]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1316 entries, 0 to 1559
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1316 non-null   object 
 1   Title             1251 non-null   object 
 2   Genre             1316 non-null   object 
 3   Release_Year      1233 non-null   Int64  
 4   Duration          1112 non-null   Int64  
 5   Rating            1212 non-null   float64
 6   IMDb_Rating       1092 non-null   float64
 7   Votes             1247 non-null   float64
 8   Director          1255 non-null   object 
 9   Country           1316 non-null   object 
 10  Language          1316 non-null   object 
 11  Budget            1213 non-null   object 
 12  Revenue           1182 non-null   object 
 13  Date_Added        1316 non-null   object 
 14  Age_Rating        1316 non-null   object 
 15  Cast              1235 non-null   object 
 16  Production_House  1252 non-null   object 
dtype

In [225]:
df['Budget'].unique()

array(['-500000', '210868644', '100147805', ..., '$80,210,058',
       '202601908', '$25,287,246'], dtype=object)

In [226]:
df['Budget'].value_counts().head(20)

,count
Budget,
-500000,43
67316469,1
15768669,1
182908079,1
76126551,1
203565907,1
221102709,1
"$194,105,905",1
108581744,1


In [227]:
df['Budget'].value_counts().tail(20)

,count
Budget,
113711529,1
$21901939,1
194517908,1
96422916,1
36043131,1
115589386,1
1616035,1
226477834,1
22771349,1


In [228]:
df['Budget'].sample(20, random_state=42)

,Budget
206,63684488
608,$119022694
393,139723780
303,45328262
189,14738991
815,$91351385
360,226768543
1187,10074376
76,$225756482
1097,139461454


In [229]:
# Remove the $ and ,, convert to a number, then set negative values to NaN.

def clean_budget(budget):
  if pd.isna(budget):
    return np.nan

  budget = str(budget).strip().replace('$', '').replace(',', '')

  try:
    budget = float(budget)
  except ValueError:
    return np.nan

  if budget < 0:
    return np.nan

  return budget

In [230]:
df['Budget'] = df['Budget'].apply(clean_budget)
df['Budget']

,Budget
0,NaN
1,210868644.0
2,100147805.0
3,95587293.0
4,8921122.0
...,...
1555,NaN
1556,61305141.0
1557,80210058.0
1558,202601908.0


✅ **Explanation:** We use a `try-except` block because after removing `$` and `,`, there might still be values that cannot be converted (like stray text). The `try-except` catches those safely and returns `NaN`. Then we apply the domain rule: budget ≥ 0.



But sometimes, because of data entry mistakes, you might have things like:

- ‎`"unknown"`

- ‎`"N/A"`

- ‎`"approx 5M"`

- ‎`"—"`

- ‎`"5000000 USD"`

All of those are called “stray text” in this context: text that doesn’t belong in a numeric budget


---
# Step 13 — Cleaning the Revenue Column

## **Problem Found**

The `Revenue` column has:

1. **Commas in numbers** — e.g. `"687,371,362"`
2. **Revenue = 0** — which is unrealistic for a movie that made it onto Netflix

## **Why is this a problem?**

Revenue of 0 is almost certainly a data entry error. A movie that generated no money whatsoever is not meaningful in most analyses.

## **How can we solve it?**

Remove commas, convert to float, then set revenue <= 0 to `NaN`.

In [235]:
df['Revenue']

,Revenue
0,1.505957e+09
1,6.873714e+08
2,4.783029e+08
3,1.295145e+09
4,6.877799e+08
...,...
1555,7.302999e+08
1556,4.270307e+07
1557,1.724231e+09
1558,NaN


In [231]:
df[df['Revenue'] == 0]

,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House


In [232]:
# 1. Convert to numeric (creates a new column so you don't overwrite yet)
df["Revenue_num"] = pd.to_numeric(df["Revenue"].astype(str).str.replace(",", ""), errors="coerce")

# 2. See all negative revenues
df[df["Revenue_num"] < 0]

,Movie_ID,Title,Genre,Release_Year,Duration,Rating,IMDb_Rating,Votes,Director,Country,Language,Budget,Revenue,Date_Added,Age_Rating,Cast,Production_House,Revenue_num


In [233]:
def clean_revenue(revenue):

  # Remove commas
  revenue = str(revenue).replace(',', '')

  # Convert to float
  revenue = float(revenue)

  # Set 0 or negative to NaN
  if revenue <= 0:
    return np.nan
  else:
    return revenue

In [234]:
df['Revenue'] = df['Revenue'].apply(clean_revenue)
df['Revenue']

,Revenue
0,1.505957e+09
1,6.873714e+08
2,4.783029e+08
3,1.295145e+09
4,6.877799e+08
...,...
1555,7.302999e+08
1556,4.270307e+07
1557,1.724231e+09
1558,NaN


✅ **Explanation:** The logic is the same as Budget, with one addition: revenue = 0 is also an outlier. We use `<= 0` instead of `< 0` to catch both zero and negative values.


---
# Step 14 — Standardising the Date_Added Column

## **Problem Found**

The `Date_Added` column stores dates in at least **four different formats**:

- `"2024-05-01"` — ISO format (Year-Month-Day)
- `"01/05/2024"` — Day/Month/Year
- `"May 1, 2024"` — Month name Day, Year
- `"20240501"` — Compact format (no separators)

## **Why is this a problem?**

You cannot sort or compare dates stored as text in different formats. `"20230101"` and `"January 1, 2023"` are the same date but Python cannot know that.

## **How can we solve it?**

We use `pd.to_datetime()` with `infer_datetime_format=True`, which is smart enough to recognise most formats automatically. For the compact format (`"20230101"`), we add a special pre-processing step.

In [237]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1316 entries, 0 to 1559
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Movie_ID          1316 non-null   object 
 1   Title             1251 non-null   object 
 2   Genre             1316 non-null   object 
 3   Release_Year      1233 non-null   Int64  
 4   Duration          1112 non-null   Int64  
 5   Rating            1212 non-null   float64
 6   IMDb_Rating       1092 non-null   float64
 7   Votes             1247 non-null   float64
 8   Director          1255 non-null   object 
 9   Country           1316 non-null   object 
 10  Language          1316 non-null   object 
 11  Budget            1170 non-null   float64
 12  Revenue           1144 non-null   float64
 13  Date_Added        1316 non-null   object 
 14  Age_Rating        1316 non-null   object 
 15  Cast              1235 non-null   object 
 16  Production_House  1252 non-null   object 
 17  

In [236]:
df['Date_Added']

,Date_Added
0,2015-10-14
1,20220620
2,20180120
3,20210103
4,2021-12-08
...,...
1555,28/10/2015
1556,"November 26, 2022"
1557,"May 19, 2018"
1558,12/11/2021


In [238]:
df['Date_Added'] = pd.to_datetime(df['Date_Added'], format='mixed', errors='coerce')

In [239]:
df['Date_Added']

,Date_Added
0,2015-10-14
1,2022-06-20
2,2018-01-20
3,2021-01-03
4,2021-12-08
...,...
1555,2015-10-28
1556,2022-11-26
1557,2018-05-19
1558,2021-12-11


In [233]:
pd.to_datetime(df['Date_Added'], format='%d-%m-%Y', errors='coerce')

✅ **Explanation:** The compact format (`"20240501"`) is the tricky one — pandas cannot detect it automatically. We handle it by checking if the string is 8 digits long, then inserting dashes to convert it to `"2024-05-01"`. After that, `pd.to_datetime()` handles everything else.


---
# Step 15 — Standardising the Country Column

## **Problem Found**

The same countries appear with different names:

- `"USA"`, `"United States"`, `"U.S."`, `"US"`, `"U.S.A"` — all mean the United States
- `"UK"`, `"United Kingdom"`, `"Britain"`, `"U.K."` — all mean the United Kingdom

## **Why is this a problem?**

If we group movies by country, the USA will appear as 5 separate groups instead of one. We will grossly undercount American movies.

## **How can we solve it?**

We build a **dictionary** that maps every known variant to the official name. Then we write a function that looks up each value in the dictionary.

✅ **Explanation:** A dictionary is the perfect tool for this problem. We store every known variant as a key (all lowercase) and the correct name as the value. Converting the input to lowercase before lookup means we handle `"USA"`, `"usa"`, and `"Usa"` all the same way.


---
# Step 16 — Standardising the Language Column

## **Problem Found**

The `Language` column has:

- Inconsistent casing: `"english"`, `"ENGLISH"`, `"English"`
- Abbreviations: `"Eng"`, `"Hin"`, `"Kor"`, `"Ger"`, `"Fre"`, `"Jap"`, `"Spa"`, `"Ita"`

## **Why is this a problem?**

`"english"` and `"ENGLISH"` will be counted as different languages. Abbreviations like `"Eng"` might not even be recognised in reports.

## **How can we solve it?**

Build a dictionary mapping abbreviations and incorrect casings to the full, properly-capitalised language name.


✅ **Explanation:** The same dictionary-lookup pattern we used for Country works perfectly here. All abbreviations like `"Hin"` become `"Hindi"` and all casing variants like `"ENGLISH"` become `"English"`.


---
# Step 16 — Standardising the Language Column

## **Problem Found**

The `Language` column has:

- Inconsistent casing: `"english"`, `"ENGLISH"`, `"English"`
- Abbreviations: `"Eng"`, `"Hin"`, `"Kor"`, `"Ger"`, `"Fre"`, `"Jap"`, `"Spa"`, `"Ita"`

## **Why is this a problem?**

`"english"` and `"ENGLISH"` will be counted as different languages. Abbreviations like `"Eng"` might not even be recognised in reports.

## **How can we solve it?**

Build a dictionary mapping abbreviations and incorrect casings to the full, properly-capitalised language name.

✅ **Explanation:** We reuse the same simple function for all three columns. It strips whitespace and converts empty strings to `NaN`, so all missing values are represented the same way.


---
# Step 18 — Final Duplicate Check (Near-Duplicate Rows)

## **Problem Found**

Even after removing duplicate Movie_IDs, the dataset may still contain rows that are very similar — the same title and year but with slightly different other values.

## **Why is this a problem?**

If the same movie appears twice with slightly different data, it will be double-counted in analysis.

## **How can we solve it?**

We check for rows where both `Title` AND `Release_Year` are the same. For any such group, we keep only the first row.

✅ **Explanation:** `drop_duplicates(subset=["Title", "Release_Year"])` looks at both columns together. Two rows must share both the same title AND the same year to be considered duplicates. This avoids accidentally removing movies that share a title but are from different years (like sequels or remakes).


---
# Step 19 — Filling Remaining Missing Values

## **Problem Found**

After all the cleaning above, some columns still have missing values. We need a strategy for each one.

## Strategy by column

| Column | Strategy | Reason |
|---|---|---|
| `Duration` | Fill with median | Median is not affected by outliers |
| `Rating` | Fill with median | Same reason |
| `IMDb_Rating` | Fill with median | Same reason |
| `Votes` | Fill with median | Same reason |
| `Budget` | Leave as NaN | Cannot guess budget — too important to fake |
| `Revenue` | Leave as NaN | Same |
| `Release_Year` | Leave as NaN | Wrong year is worse than no year |
| `Director`, `Cast`, `Production_House` | Leave as NaN | Cannot guess names |

✅ **Explanation:** We use `np.median()` to compute the median from a NumPy array. The median is better than the mean for filling missing values because it is not pulled by extreme values. We intentionally leave Budget, Revenue, and year-related columns as `NaN` because guessing wrong values would be worse than having no value.


---
# Step 20 — Final Data Type Conversion

## **Problem Found**

Some columns that should be integers (like `Release_Year`, `Votes`, `Duration`) are still stored as floats because of the `NaN` values we introduced during cleaning. Pandas requires integer columns to have no missing values.

## **How can we solve it?**

We use pandas **nullable integer type** (`"Int64"`) which allows integers with missing values. For columns with no missing values, we convert to standard `int`.

✅ **Explanation:** `"Int64"` (capital I) is pandas' nullable integer type. It can store whole numbers AND `NaN` at the same time, which regular Python `int` cannot do.


---
# Step 21 — Final Verification

Now that all cleaning is done, let's verify that our dataset is truly clean.


---
# Step 22 — Before vs After Comparison

Let's compare key statistics from the raw dataset to the cleaned dataset.


---
# Step 23 — Save the Cleaned Dataset

Finally, we save the cleaned dataset as a new CSV file so the original messy file is preserved.


---

# 🎉 Project Complete!

Here is a summary of everything we cleaned in this project:

| Step | Problem | Solution |
|------|---------|----------|
| 4 | Duplicate Movie_IDs | `drop_duplicates(subset="Movie_ID")` |
| 5 | Messy titles (spaces, newlines, caps, long) | Custom `clean_title()` function |
| 6 | Inconsistent genre formats | Custom `clean_genre()` + dictionary |
| 7 | Release year outliers (1890, 2099) | Domain-rule validation |
| 8 | Duration in mixed formats | Custom `convert_duration()` function |
| 9 | Rating stored as string | `pd.to_numeric()` |
| 10 | IMDb ratings outside 0–10 | Domain-rule validation |
| 11 | Votes with commas | Remove commas, convert to int |
| 12 | Budget with `$` signs and negatives | Remove `$`/`,`, check ≥ 0 |
| 13 | Revenue with commas, zero values | Remove `,`, check > 0 |
| 14 | Dates in 4 different formats | `pd.to_datetime()` + compact format handler |
| 15 | Country name variants | Lookup dictionary |
| 16 | Language abbreviations and casing | Lookup dictionary |
| 17 | Missing Director/Cast/Production House | Standardised to `NaN` |
| 18 | Near-duplicate rows (same Title + Year) | `drop_duplicates(subset=["Title","Release_Year"])` |
| 19 | Remaining missing numeric values | Fill with median |
| 20 | Wrong data types | `astype()` and `"Int64"` |

**Key Python concepts used:**

- `def` functions with clear parameters
- `for` loops and `if-elif-else` conditions
- `str.strip()`, `str.replace()`, `str.split()`, `str.lower()`, `str.title()`
- Dictionaries for lookup tables
- NumPy arrays and aggregate functions
- `pd.to_numeric()`, `pd.to_datetime()`
- `df.apply()` to run a function on every row
- `df.dropna()`, `df.fillna()`, `df.drop_duplicates()`
